# 05 - XGBoost + SMOTE

Th? nghi?m XGBoost kh?ng x? l? imbalance v? XGBoost sau SMOTE tr?n train set.


In [1]:

# ============================================
# SETUP: Imports + Load processed data
# Run order: 01_eda -> 02_feature_engineering -> 03_prepare_dataset -> 04_baseline_models -> 05_extension_smote_xgboost
# ============================================
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
import time
import sys

from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data'
results_dir = PROJECT_ROOT / 'results'

processed_path = DATA_DIR / 'processed_split.pkl'
results_dir.mkdir(exist_ok=True)

if not processed_path.exists():
    raise FileNotFoundError(
        'Missing ../data/processed_split.pkl. Run 03_prepare_dataset.ipynb first.'
    )

sys.path.append(str(PROJECT_ROOT / 'src'))
from evaluation import evaluate_model, print_metrics_table

# Load ??ng dict ?? l?u t? 03_prepare_dataset.ipynb.
data = joblib.load(processed_path)
X_train, X_test = data['X_train'], data['X_test']
y_train, y_test = data['y_train'], data['y_test']
# XGBoost l? tree-based -> d?ng b?n KH?NG scale.

print(f'Du lieu da load: X_train {X_train.shape}, X_test {X_test.shape}')
print(f'Train fraud rate: {y_train.mean():.4f}, Test fraud rate: {y_test.mean():.4f}')
print(f'Columns: {list(X_train.columns)}')

xgb_params = dict(
    random_state=42,
    eval_metric='logloss',
    tree_method='hist',
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    n_jobs=-1,
)


Du lieu da load: X_train (5090096, 15), X_test (1272524, 15)
Train fraud rate: 0.0013, Test fraud rate: 0.0013
Columns: ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'errorBalanceOrig', 'errorBalanceDest', 'balance_change_orig', 'balance_change_dest', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']


In [2]:

# ============================================
# XGBoost - chua xu ly imbalance
# ============================================
start = time.time()
xgb_plain = XGBClassifier(**xgb_params)
xgb_plain.fit(X_train, y_train)
xgb_plain_train_time = time.time() - start
print(f'Thoi gian train XGBoost (plain): {xgb_plain_train_time:.2f} giay')

xgb_plain_results = evaluate_model(
    xgb_plain, X_test, y_test, 'XGBoost (no imbalance handling)', xgb_plain_train_time
)
xgb_plain_results['evaluation_scope'] = 'full_test'
print(xgb_plain_results)


Thoi gian train XGBoost (plain): 40.28 giay


{'model': 'XGBoost (no imbalance handling)', 'accuracy': 0.9999960708010223, 'precision': 0.999390243902439, 'recall': 0.9975654290931223, 'f1': 0.9984770027413951, 'roc_auc': 0.9999263576308355, 'pr_auc': 0.9987268241687844, 'train_time_sec': 40.277, 'predict_time_sec': 3.102, 'evaluation_scope': 'full_test'}


In [3]:

# ============================================
# SMOTE - chi ap dung tren train set, KHONG dung tren test set
# ============================================
from imblearn.over_sampling import SMOTE

print('Truoc SMOTE:')
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print('\nSau SMOTE (chi tren train):')
print(pd.Series(y_train_smote).value_counts())

start = time.time()
xgb_smote = XGBClassifier(**xgb_params)
xgb_smote.fit(X_train_smote, y_train_smote)
xgb_smote_train_time = time.time() - start
print(f'\nThoi gian train XGBoost+SMOTE: {xgb_smote_train_time:.2f} giay')

xgb_smote_results = evaluate_model(
    xgb_smote, X_test, y_test, 'XGBoost + SMOTE', xgb_smote_train_time
)
xgb_smote_results['evaluation_scope'] = 'full_test'
print(xgb_smote_results)


Truoc SMOTE:
isFraud
0    5083526
1       6570
Name: count, dtype: int64



Sau SMOTE (chi tren train):
isFraud
0    5083526
1    5083526
Name: count, dtype: int64



Thoi gian train XGBoost+SMOTE: 87.37 giay


{'model': 'XGBoost + SMOTE', 'accuracy': 0.9999072709041245, 'precision': 0.9344729344729344, 'recall': 0.9981740718198417, 'f1': 0.9652736904061212, 'roc_auc': 0.9997667863533619, 'pr_auc': 0.998587019058034, 'train_time_sec': 87.373, 'predict_time_sec': 4.696, 'evaluation_scope': 'full_test'}


In [4]:

# ============================================
# So sanh: XGBoost plain vs XGBoost + SMOTE
# ============================================
xgb_comparison = print_metrics_table([xgb_plain_results, xgb_smote_results])
print('Bang so sanh XGBoost (plain) vs XGBoost + SMOTE:')
print(xgb_comparison)

baseline_path = results_dir / 'baseline_results.csv'
if baseline_path.exists():
    baseline_comparison = pd.read_csv(baseline_path)
    print('\nBaseline LR/KNN/DT/RF da luu tu notebook 04:')
    print(baseline_comparison)
else:
    print('\nChua thay ../results/baseline_results.csv. Hay chay 04_baseline_models.ipynb truoc de co bang baseline.')

f1_plain = xgb_plain_results['f1']
f1_smote = xgb_smote_results['f1']
recall_plain = xgb_plain_results['recall']
recall_smote = xgb_smote_results['recall']
pr_auc_plain = xgb_plain_results['pr_auc']
pr_auc_smote = xgb_smote_results['pr_auc']

print(f'\nChenh lech F1 (SMOTE - plain): {f1_smote - f1_plain:+.4f}')
print(f'Chenh lech Recall (SMOTE - plain): {recall_smote - recall_plain:+.4f}')
print(f'Chenh lech PR-AUC (SMOTE - plain): {pr_auc_smote - pr_auc_plain:+.4f}')


Bang so sanh XGBoost (plain) vs XGBoost + SMOTE:
                             model  accuracy  precision  recall      f1  \
0  XGBoost (no imbalance handling)    1.0000     0.9994  0.9976  0.9985   
1                  XGBoost + SMOTE    0.9999     0.9345  0.9982  0.9653   

   roc_auc  pr_auc  train_time_sec  predict_time_sec evaluation_scope  
0   0.9999  0.9987          40.277             3.102        full_test  
1   0.9998  0.9986          87.373             4.696        full_test  

Baseline LR/KNN/DT/RF da luu tu notebook 04:
                 model  accuracy  precision  recall      f1  roc_auc  pr_auc  \
0  Logistic Regression    0.9493     0.0239  0.9610  0.0467   0.9908  0.5806   
1                  KNN    0.9991     1.0000  0.3231  0.4884   0.8073  0.4906   
2        Decision Tree    1.0000     0.9964  0.9976  0.9970   0.9988  0.9939   
3        Random Forest    1.0000     1.0000  0.9976  0.9988   0.9997  0.9982   

   train_time_sec  predict_time_sec                          e

In [5]:

# ============================================
# Luu models + bang ket qua cho notebook 06
# ============================================
joblib.dump(xgb_plain, results_dir / 'xgb_plain_model.pkl')
joblib.dump(xgb_smote, results_dir / 'xgb_smote_model.pkl')
joblib.dump({'xgb_plain': xgb_plain, 'xgb_smote': xgb_smote}, results_dir / 'xgb_models.pkl')
print('Da luu 2 model vao ../results/')

xgb_comparison.to_csv(results_dir / 'xgb_vs_xgb_smote_comparison.csv', index=False)
joblib.dump(xgb_comparison, results_dir / 'xgb_vs_xgb_smote_comparison.pkl')
print('Da luu bang so sanh XGBoost vao ../results/xgb_vs_xgb_smote_comparison.csv')

smote_summary = pd.Series(y_train_smote).value_counts().rename_axis('isFraud').reset_index(name='count')
smote_summary.to_csv(results_dir / 'train_smote_summary.csv', index=False)
print('Da luu tom tat du lieu SMOTE vao ../results/train_smote_summary.csv')

print('\n=== TOM TAT DE VIET REPORT ===')
print(f"XGBoost (plain)   - F1: {f1_plain:.4f}, Recall: {recall_plain:.4f}, PR-AUC: {pr_auc_plain:.4f}")
print(f"XGBoost + SMOTE   - F1: {f1_smote:.4f}, Recall: {recall_smote:.4f}, PR-AUC: {pr_auc_smote:.4f}")
print(f"SMOTE cai thien Recall: {'CO' if recall_smote > recall_plain else 'KHONG'} "
      f"({recall_smote - recall_plain:+.4f})")


Da luu 2 model vao ../results/
Da luu bang so sanh XGBoost vao ../results/xgb_vs_xgb_smote_comparison.csv


Da luu tom tat du lieu SMOTE vao ../results/train_smote_summary.csv

=== TOM TAT DE VIET REPORT ===
XGBoost (plain)   - F1: 0.9985, Recall: 0.9976, PR-AUC: 0.9987
XGBoost + SMOTE   - F1: 0.9653, Recall: 0.9982, PR-AUC: 0.9986
SMOTE cai thien Recall: CO (+0.0006)
